# AF2 uniform model soup — validation only
Merata-ratakan checkpoint AF2 seed 42/123/2026 dengan bobot tetap 1/3. Tidak ada training dan test tidak dibuka.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, sys
from pathlib import Path
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'
BRANCH='codex/af2-uniform-model-soup'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result=subprocess.run(clone)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
else: raise RuntimeError('Git clone gagal tiga kali.')
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src'))
os.chdir(REPO)
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
print('REPO:',REPO,'| BRANCH:',BRANCH)

In [ ]:
import tarfile, torch
AF2_42='experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt'
AF2_123='experiments/faruq-v3-af2-igem-paired-confirmation-v1/AF2/AF2_seed123/weights/best.pt'
AF2_2026='experiments/faruq-v3-af2-igem-paired-confirmation-v1/AF2/AF2_seed2026/weights/best.pt'
CONFIRM='experiments/faruq-v3-af2-igem-paired-confirmation-v1/val_reports/af2_igem_paired_confirmation.json'
REQUIRED=('bundles/faruq-development-v3-grouped.tar',AF2_42,AF2_123,AF2_2026,CONFIRM)
PROJECT=resolve_drive_project_root(required_relative_paths=REQUIRED)
ART=[require_project_artifact(PROJECT,path) for path in REQUIRED]
ARCHIVE,*REST=ART; C42,C123,C2026,CONFIRMATION=REST
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert (DATA/'data.yaml').is_file(),DATA
assert not (DATA/'test').exists(),'Test tidak boleh tersedia.'
GROUPED=DATA/'faruq_grouped_summary.json'
OUTPUT=PROJECT/'experiments/faruq-v3-af2-uniform-soup-v1'
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('PROJECT:',PROJECT); print('OUTPUT:',OUTPUT)

In [ ]:
import json
LOG=OUTPUT/'af2_uniform_soup_run.log'; LOG.parent.mkdir(parents=True,exist_ok=True)
device='0' if torch.cuda.is_available() else 'cpu'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_uniform_soup',
 '--data-root',str(DATA),'--grouped-summary',str(GROUPED),
 '--confirmation-summary',str(CONFIRMATION),'--checkpoints',str(C42),str(C123),str(C2026),
 '--output-root',str(OUTPUT),'--device',device]
print('MENJALANKAN VALIDATION-ONLY MODEL SOUP')
with LOG.open('w',encoding='utf-8') as stream:
    result=subprocess.run(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
tail='\n'.join(LOG.read_text(errors='replace').splitlines()[-100:]); print(tail)
if result.returncode: raise RuntimeError(f'Model soup gagal: {result.returncode}; log={LOG}')
SUMMARY=OUTPUT/'val_reports/af2_uniform_soup_decision.json'
payload=json.loads(SUMMARY.read_text())
print('REFERENCE:',payload['reference_af2_three_seed_mean'])
print('SOUP     :',payload['soup_metrics'])
print('DELTAS   :',payload['deltas'])
print('CRITERIA :',payload['criteria'])
print('DECISION :',payload['decision'])
print('SUMMARY  :',SUMMARY)